# Running Single Inference

In [1]:
import yaml

from vlm_utils import load_model
from inference import run_inference

/notebooks/ML3D-VLM_based_2D_mask_grouping/Project/Milestone_4/qwen_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
with open("config.yaml") as f:
    config = yaml.safe_load(f)

In [6]:
model, processor = load_model(config)

INFO:vlm_utils:Using device: cuda
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-VL-8B-Instruct/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-VL-8B-Instruct/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b/model.safetensors.index.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-VL-8B-Instruct/revision/main "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 750/750 [00:02<0

In [7]:
from load_data import load_frame_view, RenderedSample
MASKS_ROOT = "../"          # path to your replica_masks/ directory
RGB_ROOT   = "../data/replica/"     # path to Replica scene with color/ subdir
SCENE      = "room0"

In [8]:
frame0 = load_frame_view(SCENE, 0, masks_root=MASKS_ROOT, rgb_root=RGB_ROOT)
frame50 = load_frame_view(SCENE, 50, masks_root=MASKS_ROOT, rgb_root=RGB_ROOT)
frame_pos = load_frame_view(SCENE, 10, masks_root=MASKS_ROOT, rgb_root=RGB_ROOT)
frame_neg = load_frame_view(SCENE, 80, masks_root=MASKS_ROOT, rgb_root=RGB_ROOT)

### Pair Only

In [9]:
# Pick any two mask IDs present in this frame
sample_pair_only = RenderedSample(
    condition="pair_only",
    frame_a=frame0,
    mask_id_a=8,
    frame_b=frame50,
    mask_id_b=6
)

In [11]:
# sample already built — just run it
result = run_inference(sample_pair_only, model, processor, max_new_tokens=config["inference"]["max_new_tokens"])
print(result)

{'decision': 1, 'confidence': 1.0, 'reasoning': 'Both regions outline the same wooden sideboard with identical shape, texture, and items on top, confirming it is the same physical object.'}


### Pair with Candidate

In [12]:
sample_with_candidate = RenderedSample(
    condition="pair_with_candidate",
    frame_a=frame0,
    mask_id_a=8,
    frame_b=frame50,
    mask_id_b=6, 
    observer=frame_pos,
    candidate_id=6,
)


In [13]:
# sample already built — just run it
result = run_inference(sample_with_candidate, model, processor, max_new_tokens=config["inference"]["max_new_tokens"])
print(result)

{'decision': 1, 'confidence': 1.0, 'reasoning': 'Both region A and region B are parts of the same wooden cabinet, as they are fully enclosed within the single green detection candidate in the observer frame.'}


### Pair with Context

In [14]:
sample_with_context = RenderedSample(
    condition="pair_with_context",
    frame_a=frame0,
    mask_id_a=8,
    frame_b=frame50,
    mask_id_b=6, 
    observer=frame_pos
)

In [15]:
result = run_inference(sample_with_context, model, processor, max_new_tokens=config["inference"]["max_new_tokens"])
print(result)

{'decision': 1, 'confidence': 1.0, 'reasoning': 'Both regions outline the same wooden sideboard with identical shape, position, and decorative items on top.'}
